# 🏎️ Nova Abordagem — Treinamento SAC para Trackmania

**Soft Actor-Critic (SAC)** do zero, com:
- Ator com reparametrização (log_std aprendido)
- Crítico duplo (Twin Q-Networks) para reduzir overestimation
- Target networks com Polyak averaging
- Replay Buffer eficiente
- Reward shaping progressivo
- Salvamento automático de checkpoints

---

## 1. Imports e Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Normal
from collections import deque
import random
import matplotlib.pyplot as plt
import os
import time
import copy

from tmrl.envs import GenericGymEnv
from tmrl.config.config_objects import CONFIG_DICT

# Detecta GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configurações e Hiperparâmetros

In [ ]:
# ============================================================
# HIPERPARÂMETROS — ajuste aqui antes de rodar!
# ============================================================

# --- Replay Buffer ---
TAMANHO_MEMORIA = 100_000      # Capacidade do replay buffer
TAMANHO_BATCH = 256            # Tamanho do mini-batch para treino

# --- Treinamento ---
MAX_EPISODIOS = 5000           # Número máximo de episódios
MAX_PASSOS_POR_EPISODIO = 2000 # Passos máximos por episódio
PASSOS_ANTES_DE_TREINAR = 1000 # Passos aleatórios antes de começar a treinar
TREINOS_POR_PASSO = 1          # Updates de gradiente por passo do ambiente

# --- SAC ---
GAMMA = 0.99                   # Fator de desconto
TAU = 0.005                    # Polyak averaging (soft update) das target networks
LR_ATOR = 3e-4                 # Learning rate do ator
LR_CRITICO = 3e-4              # Learning rate dos críticos
LR_ALPHA = 3e-4                # Learning rate do coeficiente de entropia
ALPHA_INICIAL = 0.2            # Coeficiente de entropia inicial
APRENDER_ALPHA = True          # Se True, alpha é ajustado automaticamente

# --- Rede Neural ---
TAMANHO_HIDDEN = 256           # Neurônios por camada hidden
LOG_STD_MIN = -20              # Limite inferior do log(std)
LOG_STD_MAX = 2                # Limite superior do log(std)

# --- Checkpoints ---
SALVAR_A_CADA = 50             # Salvar pesos a cada N episódios
PASTA_PESOS = "pesos_nova_abordagem"

# --- Observação ---
# obs = (velocidade, lidar, acao_t-1, acao_t-2)
# velocidade: shape (1,) -> 1 valor
# lidar: shape (4, 19) -> 4 frames * 19 raios = 76 valores
# acao_t-1: shape (3,) -> 3 valores
# acao_t-2: shape (3,) -> 3 valores
# TOTAL: 1 + 76 + 3 + 3 = 83
TAMANHO_INPUT = 83
TAMANHO_OUTPUT = 3  # (acelerar, frear, virar) todos em [-1, 1]

# Entropia alvo para ajuste automático de alpha
ENTROPIA_ALVO = -TAMANHO_OUTPUT  # Heurística: -dim(ação)

print("Hiperparâmetros configurados!")
print(f"  Input: {TAMANHO_INPUT} | Output: {TAMANHO_OUTPUT}")
print(f"  Memória: {TAMANHO_MEMORIA:,} | Batch: {TAMANHO_BATCH}")
print(f"  γ={GAMMA} | τ={TAU} | α₀={ALPHA_INICIAL}")

## 3. Funções Utilitárias

In [ ]:
os.makedirs(PASTA_PESOS, exist_ok=True)

def extrair_estado(obs):
    """Converte a observação do ambiente em um vetor flat numpy."""
    velocidade = np.array(obs[0]).flatten()
    lidar = np.array(obs[1]).flatten()
    acoes_anteriores = [np.array(a).flatten() for a in obs[2:]]
    historico_acoes = np.concatenate(acoes_anteriores) if len(acoes_anteriores) > 0 else np.array([])
    return np.concatenate([velocidade, lidar, historico_acoes])


def soft_update(target, source, tau):
    """Polyak averaging: θ_target = τ*θ_source + (1-τ)*θ_target"""
    for tp, sp in zip(target.parameters(), source.parameters()):
        tp.data.copy_(tau * sp.data + (1.0 - tau) * tp.data)


print("Utilitários carregados!")

## 4. Replay Buffer

In [ ]:
class ReplayBuffer:
    """Buffer de experiências para off-policy learning."""
    
    def __init__(self, capacidade, dim_estado, dim_acao):
        self.capacidade = capacidade
        self.ponteiro = 0
        self.tamanho = 0
        
        # Pré-alocação para eficiência
        self.estados = np.zeros((capacidade, dim_estado), dtype=np.float32)
        self.acoes = np.zeros((capacidade, dim_acao), dtype=np.float32)
        self.recompensas = np.zeros(capacidade, dtype=np.float32)
        self.prox_estados = np.zeros((capacidade, dim_estado), dtype=np.float32)
        self.dones = np.zeros(capacidade, dtype=np.float32)
    
    def guardar(self, estado, acao, recompensa, prox_estado, done):
        idx = self.ponteiro % self.capacidade
        self.estados[idx] = estado
        self.acoes[idx] = acao
        self.recompensas[idx] = recompensa
        self.prox_estados[idx] = prox_estado
        self.dones[idx] = float(done)
        self.ponteiro += 1
        self.tamanho = min(self.tamanho + 1, self.capacidade)
    
    def amostrar(self, batch_size):
        indices = np.random.randint(0, self.tamanho, size=batch_size)
        return (
            torch.tensor(self.estados[indices], dtype=torch.float32).to(device),
            torch.tensor(self.acoes[indices], dtype=torch.float32).to(device),
            torch.tensor(self.recompensas[indices], dtype=torch.float32).to(device),
            torch.tensor(self.prox_estados[indices], dtype=torch.float32).to(device),
            torch.tensor(self.dones[indices], dtype=torch.float32).to(device),
        )
    
    def __len__(self):
        return self.tamanho


print("ReplayBuffer definido!")

## 5. Redes Neurais (Ator + Crítico Duplo)

In [ ]:
class AtorSAC(nn.Module):
    """Ator estocástico com reparametrização (Gaussian policy).
    
    Saída: média (mu) e log(std) para cada dimensão da ação.
    Ação é amostrada via reparametrização e passada por tanh para [-1, 1].
    """
    
    def __init__(self, dim_estado, dim_acao, hidden_size=256):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Linear(dim_estado, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
        )
        self.camada_mu = nn.Linear(hidden_size, dim_acao)
        self.camada_log_std = nn.Linear(hidden_size, dim_acao)
    
    def forward(self, estado):
        features = self.rede(estado)
        mu = self.camada_mu(features)
        log_std = self.camada_log_std(features)
        log_std = torch.clamp(log_std, LOG_STD_MIN, LOG_STD_MAX)
        return mu, log_std
    
    def amostrar_acao(self, estado):
        """Amostra uma ação com reparametrização e calcula log_prob."""
        mu, log_std = self.forward(estado)
        std = log_std.exp()
        
        # Reparametrização: ação = tanh(mu + std * epsilon)
        distribuicao = Normal(mu, std)
        x_t = distribuicao.rsample()  # rsample para backprop
        acao = torch.tanh(x_t)
        
        # Correção do log_prob para a transformação tanh
        log_prob = distribuicao.log_prob(x_t)
        log_prob -= torch.log(1 - acao.pow(2) + 1e-6)
        log_prob = log_prob.sum(dim=-1, keepdim=True)
        
        return acao, log_prob
    
    def acao_deterministica(self, estado):
        """Ação determinística (para avaliação)."""
        mu, _ = self.forward(estado)
        return torch.tanh(mu)


class CriticoDuploSAC(nn.Module):
    """Dois Q-networks independentes (Twin Q) para reduzir overestimation.
    
    Cada Q(s, a) recebe estado+ação concatenados e retorna um valor escalar.
    """
    
    def __init__(self, dim_estado, dim_acao, hidden_size=256):
        super().__init__()
        dim_input = dim_estado + dim_acao
        
        self.q1 = nn.Sequential(
            nn.Linear(dim_input, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1),
        )
        
        self.q2 = nn.Sequential(
            nn.Linear(dim_input, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1),
        )
    
    def forward(self, estado, acao):
        sa = torch.cat([estado, acao], dim=-1)
        return self.q1(sa), self.q2(sa)


print("Redes neurais definidas!")

## 6. Agente SAC

In [ ]:
class AgenteSAC:
    """Agente Soft Actor-Critic completo."""
    
    def __init__(self):
        # Ator
        self.ator = AtorSAC(TAMANHO_INPUT, TAMANHO_OUTPUT, TAMANHO_HIDDEN).to(device)
        self.otimizador_ator = optim.Adam(self.ator.parameters(), lr=LR_ATOR)
        
        # Críticos (online + target)
        self.criticos = CriticoDuploSAC(TAMANHO_INPUT, TAMANHO_OUTPUT, TAMANHO_HIDDEN).to(device)
        self.criticos_target = copy.deepcopy(self.criticos).to(device)
        self.otimizador_criticos = optim.Adam(self.criticos.parameters(), lr=LR_CRITICO)
        
        # Congela target (não recebe gradientes)
        for p in self.criticos_target.parameters():
            p.requires_grad = False
        
        # Alpha (coeficiente de entropia)
        if APRENDER_ALPHA:
            self.log_alpha = torch.tensor(np.log(ALPHA_INICIAL), dtype=torch.float32,
                                          requires_grad=True, device=device)
            self.otimizador_alpha = optim.Adam([self.log_alpha], lr=LR_ALPHA)
            self.entropia_alvo = ENTROPIA_ALVO
        else:
            self.log_alpha = torch.tensor(np.log(ALPHA_INICIAL), dtype=torch.float32,
                                          device=device)
    
    @property
    def alpha(self):
        return self.log_alpha.exp()
    
    def selecionar_acao(self, estado, deterministico=False):
        """Seleciona uma ação dado um estado (numpy -> numpy)."""
        estado_tensor = torch.tensor(estado, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            if deterministico:
                acao = self.ator.acao_deterministica(estado_tensor)
            else:
                acao, _ = self.ator.amostrar_acao(estado_tensor)
        return acao.cpu().numpy().flatten()
    
    def treinar(self, buffer):
        """Executa um passo de treinamento SAC."""
        estados, acoes, recompensas, prox_estados, dones = buffer.amostrar(TAMANHO_BATCH)
        
        alpha = self.alpha.detach()
        
        # ============================
        # 1. ATUALIZAR CRÍTICOS
        # ============================
        with torch.no_grad():
            # Amostra próxima ação do ator atual
            prox_acao, prox_log_prob = self.ator.amostrar_acao(prox_estados)
            
            # Q-value target = min(Q1_target, Q2_target) - alpha * log_prob
            q1_target, q2_target = self.criticos_target(prox_estados, prox_acao)
            q_target = torch.min(q1_target, q2_target) - alpha * prox_log_prob
            
            # Bellman target
            y = recompensas.unsqueeze(-1) + GAMMA * (1 - dones.unsqueeze(-1)) * q_target
        
        # Q-values atuais
        q1_atual, q2_atual = self.criticos(estados, acoes)
        
        # Loss dos críticos (MSE)
        loss_criticos = F.mse_loss(q1_atual, y) + F.mse_loss(q2_atual, y)
        
        self.otimizador_criticos.zero_grad()
        loss_criticos.backward()
        torch.nn.utils.clip_grad_norm_(self.criticos.parameters(), max_norm=1.0)
        self.otimizador_criticos.step()
        
        # ============================
        # 2. ATUALIZAR ATOR
        # ============================
        acao_nova, log_prob_nova = self.ator.amostrar_acao(estados)
        q1_nova, q2_nova = self.criticos(estados, acao_nova)
        q_nova = torch.min(q1_nova, q2_nova)
        
        # Loss do ator: maximizar Q - alpha * log_prob
        loss_ator = (alpha * log_prob_nova - q_nova).mean()
        
        self.otimizador_ator.zero_grad()
        loss_ator.backward()
        torch.nn.utils.clip_grad_norm_(self.ator.parameters(), max_norm=1.0)
        self.otimizador_ator.step()
        
        # ============================
        # 3. ATUALIZAR ALPHA (opcional)
        # ============================
        loss_alpha = 0.0
        if APRENDER_ALPHA:
            with torch.no_grad():
                _, log_prob_alpha = self.ator.amostrar_acao(estados)
            loss_alpha = -(self.log_alpha * (log_prob_alpha + self.entropia_alvo).detach()).mean()
            
            self.otimizador_alpha.zero_grad()
            loss_alpha.backward()
            self.otimizador_alpha.step()
            loss_alpha = loss_alpha.item()
        
        # ============================
        # 4. SOFT UPDATE DAS TARGETS
        # ============================
        soft_update(self.criticos_target, self.criticos, TAU)
        
        return {
            "loss_criticos": loss_criticos.item(),
            "loss_ator": loss_ator.item(),
            "loss_alpha": loss_alpha,
            "alpha": alpha.item(),
            "q_medio": q_nova.mean().item(),
        }
    
    def salvar(self, caminho_prefixo):
        """Salva todos os pesos do agente."""
        torch.save(self.ator.state_dict(), f"{caminho_prefixo}_ator.pth")
        torch.save(self.criticos.state_dict(), f"{caminho_prefixo}_criticos.pth")
        torch.save(self.criticos_target.state_dict(), f"{caminho_prefixo}_criticos_target.pth")
        if APRENDER_ALPHA:
            torch.save(self.log_alpha, f"{caminho_prefixo}_log_alpha.pth")
        print(f"  💾 Pesos salvos: {caminho_prefixo}")
    
    def carregar(self, caminho_prefixo):
        """Carrega pesos salvos."""
        self.ator.load_state_dict(torch.load(f"{caminho_prefixo}_ator.pth", map_location=device))
        self.criticos.load_state_dict(torch.load(f"{caminho_prefixo}_criticos.pth", map_location=device))
        self.criticos_target.load_state_dict(torch.load(f"{caminho_prefixo}_criticos_target.pth", map_location=device))
        if APRENDER_ALPHA:
            self.log_alpha = torch.load(f"{caminho_prefixo}_log_alpha.pth", map_location=device)
            self.log_alpha.requires_grad = True
            self.otimizador_alpha = optim.Adam([self.log_alpha], lr=LR_ALPHA)
        print(f"  ✅ Pesos carregados: {caminho_prefixo}")


print("Agente SAC definido!")

## 7. Reward Shaping

In [ ]:
def calcular_recompensa(recompensa_crua, obs_atual, prox_obs, passo, done):
    """Aplica reward shaping para guiar o aprendizado.
    
    A recompensa crua do TMRL já é baseada em progresso na pista.
    Aqui adicionamos sinais extras para acelerar o aprendizado.
    """
    recompensa = recompensa_crua
    
    # Extrai informações da próxima observação
    velocidade = float(np.array(prox_obs[0]).flatten()[0])
    lidar_atual = np.array(prox_obs[1]).flatten()  # 4 frames * 19 raios
    lidar_recente = lidar_atual[-19:]  # Último frame de LIDAR
    menor_distancia = float(np.min(lidar_recente))
    
    # --- Bônus por velocidade (quando está progredindo) ---
    if recompensa_crua > 0:
        recompensa += velocidade * 0.01
    
    # --- Penalização por proximidade de parede ---
    if menor_distancia < 0.1 and not done:
        recompensa -= 1.0
    
    # --- Penalização por ficar parado ---
    if velocidade < 5.0 and passo > 20:
        recompensa -= 0.3
    
    # --- Penalização por episódio terminar (crash/off-track) ---
    if done and passo < 50:
        recompensa -= 5.0
    
    return recompensa


print("Reward shaping definido!")

## 8. Inicializar Ambiente e Agente

In [ ]:
# Cria o ambiente
env = GenericGymEnv(id="real-time-gym-ts-v1", gym_kwargs={"config": CONFIG_DICT})
print("Ambiente criado!")

# Cria o agente
agente = AgenteSAC()
print("Agente SAC criado!")

# Cria o replay buffer
buffer = ReplayBuffer(TAMANHO_MEMORIA, TAMANHO_INPUT, TAMANHO_OUTPUT)
print("Replay Buffer criado!")

# Históricos para gráficos
historico_recompensas = []
historico_loss_ator = []
historico_loss_criticos = []
historico_alpha = []
historico_q_medio = []
historico_passos = []

print("\n🏁 Tudo pronto para treinar!")

## 8.1 (Opcional) Carregar Pesos Salvos

Descomente e execute esta célula para retomar um treino anterior.

In [ ]:
# EPISODIO_RETOMAR = 100  # Número do checkpoint a carregar
# agente.carregar(f"{PASTA_PESOS}/checkpoint_ep{EPISODIO_RETOMAR}")
# print(f"Retomando do episódio {EPISODIO_RETOMAR}")

## 9. 🏎️ Loop de Treinamento

**IMPORTANTE**: Clique na janela do Trackmania após executar esta célula!

In [ ]:
print("="*60)
print("🏎️  INÍCIO DO TREINAMENTO SAC")
print("   Clique na janela do Trackmania AGORA!")
print("="*60)
time.sleep(3)  # Tempo para clicar na janela do jogo

total_passos = 0
melhor_recompensa = -float("inf")

try:
    for episodio in range(1, MAX_EPISODIOS + 1):
        obs, info = env.reset()
        estado = extrair_estado(obs)
        recompensa_ep = 0
        losses_ep = {"loss_ator": [], "loss_criticos": [], "alpha": [], "q_medio": []}
        
        for passo in range(MAX_PASSOS_POR_EPISODIO):
            # --- SELECIONAR AÇÃO ---
            if total_passos < PASSOS_ANTES_DE_TREINAR:
                # Ação aleatória durante aquecimento
                acao = np.random.uniform(-1.0, 1.0, size=TAMANHO_OUTPUT).astype(np.float32)
            else:
                acao = agente.selecionar_acao(estado)
            
            # --- INTERAÇÃO COM O AMBIENTE ---
            prox_obs, recompensa_crua, terminated, truncated, info = env.step(acao)
            done = terminated or truncated
            
            prox_estado = extrair_estado(prox_obs)
            
            # --- REWARD SHAPING ---
            recompensa = calcular_recompensa(recompensa_crua, obs, prox_obs, passo, done)
            recompensa_ep += recompensa
            
            # --- GUARDAR NO BUFFER ---
            buffer.guardar(estado, acao, recompensa, prox_estado, done)
            total_passos += 1
            
            # --- TREINAR ---
            if total_passos >= PASSOS_ANTES_DE_TREINAR and len(buffer) >= TAMANHO_BATCH:
                for _ in range(TREINOS_POR_PASSO):
                    info_treino = agente.treinar(buffer)
                    losses_ep["loss_ator"].append(info_treino["loss_ator"])
                    losses_ep["loss_criticos"].append(info_treino["loss_criticos"])
                    losses_ep["alpha"].append(info_treino["alpha"])
                    losses_ep["q_medio"].append(info_treino["q_medio"])
            
            if done:
                break
            
            estado = prox_estado
            obs = prox_obs
        
        # --- REGISTROS ---
        historico_recompensas.append(recompensa_ep)
        historico_passos.append(passo + 1)
        
        media_loss_ator = np.mean(losses_ep["loss_ator"]) if losses_ep["loss_ator"] else 0
        media_loss_criticos = np.mean(losses_ep["loss_criticos"]) if losses_ep["loss_criticos"] else 0
        media_alpha = np.mean(losses_ep["alpha"]) if losses_ep["alpha"] else ALPHA_INICIAL
        media_q = np.mean(losses_ep["q_medio"]) if losses_ep["q_medio"] else 0
        
        historico_loss_ator.append(media_loss_ator)
        historico_loss_criticos.append(media_loss_criticos)
        historico_alpha.append(media_alpha)
        historico_q_medio.append(media_q)
        
        # Média móvel das últimas 20 recompensas
        media_20 = np.mean(historico_recompensas[-20:]) if len(historico_recompensas) >= 20 else np.mean(historico_recompensas)
        
        # Log
        status = "🔥" if total_passos >= PASSOS_ANTES_DE_TREINAR else "⏳"
        print(f"{status} Ep {episodio:4d}/{MAX_EPISODIOS} | "
              f"Passos: {passo+1:4d} | "
              f"Rew: {recompensa_ep:7.1f} | "
              f"Média20: {media_20:7.1f} | "
              f"α: {media_alpha:.4f} | "
              f"Q̄: {media_q:.2f} | "
              f"Buffer: {len(buffer):,}")
        
        # --- SALVAR CHECKPOINT ---
        if episodio % SALVAR_A_CADA == 0:
            agente.salvar(f"{PASTA_PESOS}/checkpoint_ep{episodio}")
        
        # Salvar melhor modelo
        if recompensa_ep > melhor_recompensa:
            melhor_recompensa = recompensa_ep
            agente.salvar(f"{PASTA_PESOS}/melhor")

except KeyboardInterrupt:
    print("\n⚠️ Treinamento interrompido manualmente!")

finally:
    print("\n" + "="*60)
    print("💾 Salvando pesos finais...")
    agente.salvar(f"{PASTA_PESOS}/final")
    print(f"Total de passos: {total_passos:,}")
    print(f"Melhor recompensa: {melhor_recompensa:.1f}")
    print("="*60)

## 10. 📊 Gráficos de Treinamento

In [ ]:
def media_movel(dados, janela=20):
    """Calcula média móvel para suavizar os gráficos."""
    if len(dados) < janela:
        return dados
    ret = np.cumsum(dados)
    ret[janela:] = ret[janela:] - ret[:-janela]
    result = list(dados[:janela-1])  # Primeiros valores sem média
    result.extend(ret[janela-1:] / janela)
    return result

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
fig.suptitle("Treinamento SAC — Nova Abordagem", fontsize=16, fontweight="bold")

# 1. Recompensas
axes[0, 0].plot(historico_recompensas, alpha=0.3, color="blue", label="Por episódio")
axes[0, 0].plot(media_movel(historico_recompensas), color="blue", linewidth=2, label="Média Móvel (20)")
axes[0, 0].set_title("Recompensa por Episódio")
axes[0, 0].set_xlabel("Episódio")
axes[0, 0].set_ylabel("Recompensa")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Passos por episódio
axes[0, 1].plot(historico_passos, alpha=0.3, color="green", label="Por episódio")
axes[0, 1].plot(media_movel(historico_passos), color="green", linewidth=2, label="Média Móvel (20)")
axes[0, 1].set_title("Duração do Episódio")
axes[0, 1].set_xlabel("Episódio")
axes[0, 1].set_ylabel("Passos")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Loss do Ator
axes[0, 2].plot(historico_loss_ator, alpha=0.3, color="red", label="Por episódio")
axes[0, 2].plot(media_movel(historico_loss_ator), color="red", linewidth=2, label="Média Móvel (20)")
axes[0, 2].set_title("Loss do Ator")
axes[0, 2].set_xlabel("Episódio")
axes[0, 2].set_ylabel("Loss")
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# 4. Loss dos Críticos
axes[1, 0].plot(historico_loss_criticos, alpha=0.3, color="orange", label="Por episódio")
axes[1, 0].plot(media_movel(historico_loss_criticos), color="orange", linewidth=2, label="Média Móvel (20)")
axes[1, 0].set_title("Loss dos Críticos")
axes[1, 0].set_xlabel("Episódio")
axes[1, 0].set_ylabel("Loss")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 5. Alpha (Entropia)
axes[1, 1].plot(historico_alpha, color="purple", linewidth=2)
axes[1, 1].set_title("Alpha (Coeficiente de Entropia)")
axes[1, 1].set_xlabel("Episódio")
axes[1, 1].set_ylabel("Alpha")
axes[1, 1].grid(True, alpha=0.3)

# 6. Q-value médio
axes[1, 2].plot(historico_q_medio, alpha=0.3, color="teal", label="Por episódio")
axes[1, 2].plot(media_movel(historico_q_medio), color="teal", linewidth=2, label="Média Móvel (20)")
axes[1, 2].set_title("Q-Value Médio")
axes[1, 2].set_xlabel("Episódio")
axes[1, 2].set_ylabel("Q")
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{PASTA_PESOS}/graficos_treinamento.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Gráficos salvos em {PASTA_PESOS}/graficos_treinamento.png")